### step1 read the json file using the spark dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, FloatType
name_schema=StructType(fields=[StructField("forename", StringType(), True),
                               StructField("surname", StringType(), True)]) 
circuits_schema=StructType(fields=[StructField("circuitId", IntegerType(), False),
                                   StructField("circuit Name", StringType(), True),
                                   StructField("location", StringType(), True),
                                   StructField("country", StringType(), True),
                                   StructField("lat", FloatType(), True),
                                   StructField("lng", FloatType(), True),
                                   StructField("alt", IntegerType(), True),
                                   StructField("url", StringType(), True)])
race_schema=StructType(fields=[StructField("raceId", IntegerType(), False),
                               StructField("year", IntegerType(), True),
                               StructField("round", IntegerType(), True),
                               StructField("circuitId", IntegerType(), True),
                               StructField("name", StringType(), True),
                               StructField("date", DateType(), True),
                               StructField("time", StringType(), True),
                               StructField("url", StringType(), True)])
results_schema=StructType(fields=[StructField("resultId", IntegerType(), False),
                                  StructField("raceId", IntegerType(), True),
                                  StructField("driverId", IntegerType(), True),
                                  StructField("constructorId", IntegerType(), True),
                                  StructField("number", IntegerType(), True),
                                  StructField("grid", IntegerType(), True),
                                  StructField("position", IntegerType(), True),
                                  StructField("positionText", StringType(), True),
                                  StructField("positionOrder", IntegerType(), True),
                                  StructField("points", FloatType(), True),
                                  StructField("laps", IntegerType(), True),
                                  StructField("time", StringType(), True),
                                  StructField("milliseconds", IntegerType(), True),
                                  StructField("fastestLap", IntegerType(), True),
                                  StructField("rank", IntegerType(), True),
                                  StructField("fastestLapTime", StringType(), True),
                                  StructField("fastestLapSpeed", FloatType(), True),
                                  StructField("statusId", StringType(), True)])
drivers_schema=StructType(fields=[StructField("driverId", IntegerType(), False),
                                  StructField("driverRef", StringType(), True),
                                  StructField("number", IntegerType(), True),
                                  StructField("code", StringType(), True),
                                  StructField("name", name_schema),
                                  StructField("dob", DateType(), True),
                                  StructField("nationality", StringType(), True),
                                  StructField("url", StringType(), True)])
constructor_schema=StructType(fields=[StructField("constructorId", IntegerType(), False),
                                     StructField("constructorRef", StringType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("nationality", StringType(), True),
                                     StructField("url", StringType(), True)])
# DBTITLE 1

In [0]:
drivers_df = spark.read \
.schema(drivers_schema) \
.json("/mnt/formula1dl4dbcourse/raw/drivers.json")

drivers_df.printSchema()

In [0]:
display(drivers_df)

###step2 - rename cols & add new cols
###step3 - drop unwanted cols

In [0]:
from pyspark.sql.functions import current_timestamp, concat, col, lit
drivers_with_cols_df = drivers_df.withColumnRenamed("driverId", "driver_id") \
.withColumnRenamed("driverRef", "driver_ref") \
.withColumn("ingestion_date", current_timestamp()) \
.withColumn("name", concat(col("name.forename"), lit(" "), col("name.surname"))) \
.drop("url", "name.forename", "name.surname")
display(drivers_with_cols_df)

### step4 - write output to processed container in parquet format

In [0]:
drivers_with_cols_df.write.mode("overwrite").format("parquet").saveAsTable("f1_processed.drivers")

In [0]:
display(spark.read.parquet("/mnt/formula1dl4dbcourse/processed/drivers"))

In [0]:
dbutils.notebook.exit("Success")